In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


: 

## Fungsi Utama: `compute_structure_tensor`

Fungsi ini menerima gradient \( I_x \) dan \( I_y \), lalu menghitung struktur tensor \( M \).


In [ ]:
def compute_structure_tensor(Ix, Iy, window_size=(5, 5), use_gaussian=True, sigma=1.0):
    """
    Menghitung Struktur Matriks Harris (M) per piksel.
    
    Input:
        Ix, Iy      : gradient image (hasil dari Soal 1)
        window_size : ukuran window (tuple), contoh (5,5)
        use_gaussian: True = Gaussian window, False = Box filter
        sigma       : sigma untuk Gaussian (hanya dipakai jika use_gaussian=True)
    
    Output:
        A, B, C     : hasil windowing dari Ixx, Iyy, Ixy
                      sehingga M = [[A, C], [C, B]] di setiap piksel
    """
    # 1. Hitung produk gradien (element-wise)
    Ixx = Ix * Ix
    Iyy = Iy * Iy
    Ixy = Ix * Iy

    # 2. Windowing / Smoothing
    if use_gaussian:
        # Pastikan kernel size ganjil
        ksize = window_size[0] if window_size[0] % 2 == 1 else window_size[0] + 1
        A = cv2.GaussianBlur(Ixx, (ksize, ksize), sigma)
        B = cv2.GaussianBlur(Iyy, (ksize, ksize), sigma)
        C = cv2.GaussianBlur(Ixy, (ksize, ksize), sigma)
    else:
        # Box filter (sum dalam window, tanpa normalize)
        A = cv2.boxFilter(Ixx, -1, window_size, normalize=False)
        B = cv2.boxFilter(Iyy, -1, window_size, normalize=False)
        C = cv2.boxFilter(Ixy, -1, window_size, normalize=False)

    return A, B, C


In [ ]:
def harris_detector(image, window_size=(5, 5), k=0.04, use_gaussian=True, sigma=1.0):
    """
    Harris Corner Detector.
    Bagian M (struktur tensor) adalah tanggung jawab utama.
    """
    # Pastikan float
    if image.dtype != np.float32 and image.dtype != np.float64:
        image = image.astype(np.float32)

    # === Gradient (sementara, bisa diganti hasil dari soal nomor 1) ===
    Ix = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    Iy = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)

    # === Struktur Tensor M (bagian Raka) ===
    A, B, C = compute_structure_tensor(
        Ix, Iy,
        window_size=window_size,
        use_gaussian=use_gaussian,
        sigma=sigma
    )

    # === Response R (bagian Naufal, sementara dihitung di sini) ===
    det_M = A * B - C * C
    trace_M = A + B
    response = det_M - k * (trace_M ** 2)

    return response, A, B, C, Ix, Iy


In [ ]:
# Load gambar (ganti path sesuai file tugas)
img = cv2.imread("building.png", cv2.IMREAD_GRAYSCALE)

# Jalankan detector
response, A, B, C, Ix, Iy = harris_detector(
    img, 
    window_size=(5, 5), 
    use_gaussian=True,
    sigma=1.0
)

print("Shape citra asli :", img.shape)
print("Shape response   :", response.shape)
print("Shape A (Sxx)    :", A.shape)
print("Max response     :", response.max())
print("Min response     :", response.min())


## Visualisasi Sederhana

Tampilkan response map (semakin terang = semakin kuat kandidat sudut).


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.title("Citra Asli")
plt.imshow(img, cmap='gray')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title("Response R")
plt.imshow(response, cmap='hot')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title("A (Ixx smoothed)")
plt.imshow(A, cmap='gray')
plt.axis('off')

plt.tight_layout()
plt.show()
